In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
os.getenv('OPENAI_API_KEY')[-10:]

'4JzPhUIcUA'

## ReAct와 LangGraph의 연결 방식

ReAct는 Reason과 Act를 번갈아 수행하는 패턴이다. 질문을 받으면 먼저 현재 답변에 필요한 정보를 판단하고, 부족하면 도구를 호출한다. 그 뒤 도구 결과를 반영해 다시 생각한다.

아래 코드는 이 흐름을 `think -> act -> answer`의 순서로 분해한다. 상태에는 질문, 생각 메모, 도구 결과, 최종 답변만 둬서 흐름이 어떻게 이어지는지 쉽게 보이도록 만든다.

In [2]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class ReActState(TypedDict):
    question:str
    thought:str
    tool_result:str
    answer:str

def think(state:ReActState):
    question = state['question']
    if '문서' in question or '근거' in question:
        thought = '검색도구가 필요합니다.'
    else:
        thought = '바로 답할수 있습니다.'
    return {'thought':thought}

def act(state:ReActState):
    question = state['question']
    if '문서' in question or '근거' in question:
        tool_result = '벡터DB에서 관련 문단 2개를 찾았습니다.'
    else:
        tool_result = '외부도구가 필요하지 않다.'
    return {'tool_result':tool_result}

def answer(state:ReActState):
    answer = f"질문: {state['question']}\n생각:{state['thought']}\n도구:{state['tool_result']}"
    return {'answer':answer}

workflow = StateGraph(ReActState)
workflow.add_node('think',think)
workflow.add_node('act',act)
workflow.add_node('answer',answer)
workflow.add_edge(START,'think') # 뭔가 시작 포인트가 필요함 노드에서 상세변화를 처리하는 edge 이다
workflow.add_edge('think','act')
workflow.add_edge('act', 'answer') 
workflow.add_edge('answer',END)
app = workflow.compile()
knwargs = {
    'question' : '문서 근거가 필요한 더미 질문입니다.',
    'thought' : '',
    'tool_result': '',
    'answer':''
}
result = app.invoke(knwargs)
print(result['answer'])

질문: 문서 근거가 필요한 더미 질문입니다.
생각:검색도구가 필요합니다.
도구:벡터DB에서 관련 문단 2개를 찾았습니다.


In [5]:
import chromadb
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from openai import OpenAI
import os
import re
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

embedding_function = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-V2')
chroma_client = chromadb.PersistentClient(path='chroma_lesson_01_v3')
collection = chroma_client.get_or_create_collection(
    name='react_rag',
    embedding_function=embedding_function
)
if collection.count() == 0:
    collection.add(
        ids=["doc_langgraph", "doc_react", "doc_rag", "doc_chroma"],
        documents=[
            "LangGraph는 상태 기반 그래프로 다단계 에이전트를 설계한다.",
            "ReAct는 reasoning과 acting을 번갈아 수행해 도구 사용을 결합한다.",
            "RAG는 벡터DB 검색 결과를 근거로 답변 품질을 높인다.",
            "ChromaDB는 문서 임베딩을 저장하고 유사한 문서를 검색하는 벡터DB다.",
        ],
    )

# 크로마 DB는 지식 베이스라고 생각하면 된다.
# 그래프에 들어갈 노드 함수구성 --> Tool
class ReActRAGState(TypedDict):
    question:str
    thought:str
    action:str
    tool_result:str
    answer:str

def think(state:ReActRAGState):
    question = state['question'].lower()
    if any(keyword in question for keyword in ['근거','문서','설명','rag']):
        thought = 'CromaDB에서 근거를 먼저 찾아야 한다'
        action = 'search'
    else:
        thought = '바로 답할 수 있다'
        action = 'respond'
    return {'thought': thought, 'action': action}
def search_context(state:ReActRAGState):
    result = collection.query(query_texts=[state['question']], n_results=2)
    context = '\n'.join(result['documents'][0])
    return {'tool_result':context}

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def response(state:ReActRAGState): # LLM이 답변
    answer = f"질문: {state['question']}\n생각:{state['thought']}\n도구:{state['tool_result']}\n\n답변을 생성하세요"
    response = client.chat.completions.create(
        model='gpt-5.4-nano',
        messages=[
            {'role':'system','content':'당신은 인공지능 자연어 NLP 전문가입니다.'},
            {'role':'user','content':answer}
        ],
        temperature= 0
    )
    return  {'answer':response.choices[0].message.content.strip()}

# graph 구성
workflow = StateGraph(ReActRAGState)
workflow.add_node('think',think) # add_node 질문을 보고 바로 답할지 검색할지 선택하는 노드
workflow.add_node('search_context',search_context)
workflow.add_node('response',response) # 최종 답변을 선택함
workflow.add_edge(START,'think')
workflow.add_conditional_edges( #컨디션에 따라 분기된다.
    'think',
    lambda state : state['action'],
    {"search": 'search_context', "respond" : "response"}
)
workflow.add_edge('search_context','response')
workflow.add_edge('response',END)

app = workflow.compile()
knwargs = {
    'question' : 'LangGraph와 RAG가 연결될때 왜 ReAct는 유용한가?',
    'thought' : '',
    'action' : '',
    'tool_result': '',
    'answer':''
}
result = app.invoke(knwargs)
print('route',result['action'])
print('thought',result['thought'])
print('context')
print(result['tool_result'])
print('answer')
print(result['answer'])

route search
thought CromaDB에서 근거를 먼저 찾아야 한다
context
LangGraph는 상태 기반 그래프로 다단계 에이전트를 설계한다.
ReAct는 reasoning과 acting을 번갈아 수행해 도구 사용을 결합한다.
answer
LangGraph와 RAG를 함께 쓸 때 **ReAct가 유용한 이유**는, “근거 검색(RAG)”과 “도구 실행/추론(에이전트 행동)”을 **자연스럽게 왕복**시키면서 다단계 의사결정을 안정적으로 만들기 때문입니다.

- **RAG는 보통 ‘검색→근거 제공’에 강점**이 있지만, 그 근거를 바탕으로 **무엇을 더 확인해야 하는지**, **어떤 도구를 어떤 순서로 써야 하는지**까지는 별도의 에이전트 로직이 필요합니다.  
- **LangGraph는 상태(state)를 들고 다니며** 여러 단계를 그래프로 구성할 수 있어, “검색 결과를 보고 다음 행동을 결정”하는 흐름을 만들기 좋습니다.
- 이때 **ReAct(Reason + Act)**는
  1) 먼저 **Reasoning**으로 “지금 가진 근거로는 답이 완성되지 않았는지 / 어떤 추가 정보가 필요한지”를 판단하고  
  2) 그 판단에 따라 **Acting(도구 호출)**을 수행한 뒤  
  3) 결과를 다시 **Reasoning에 반영**하는 식으로  
  **검색과 도구 사용을 반복 루프**로 엮어줍니다.

따라서 연결 관점에서 정리하면:

1. **CromaDB(벡터DB) 같은 곳에서 근거를 먼저 찾고(RAG)**  
2. LangGraph의 단계/상태에 따라 그 근거를 바탕으로  
3. **ReAct가 “다음에 어떤 도구를 써야 하는지”를 추론하고 실행**하며  
4. 실행 결과를 다시 근거로 삼아 **추론을 갱신**합니다.

결과적으로 ReAct는 RAG의 “근거 수집”을 끝내지 않고, **근거를 실제 답으로 완성하기 위한 행동(추가 검색, 계산, 외부 도구 호출 등)**을 체계적으로 연결해 주기 때문에 LangGraph와 궁합이 좋습니다

In [ ]:
# 기업문서를 벡터Db화  pdf
from pathlib import Path
from typing import Iterable
from uuid import uuid4

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from pypdf import PdfReader

# 문서경로 지정
enterprise_folder = Path(r'C:\skn29\LangChain\06-01\documents')
# pdf 파일 읽기
def read_pdf_file(path:Path)->str:
    reader = PdfReader(str(path))
    pages:list[str]=[]

    for page in reader.pages:    
        pages.append(page.extract_text() or "")
    return '\n'.join(pages)
    
enterprise_embedding = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')
enterprize_client = chromadb.PersistentClient()
enterprise_collection = enterprize_client.get_or_create_collection(
    name='enterprise_document',
    embedding_function=enterprise_embedding,
    metadata={'hnsw:space':'cosine'}
)
# txt, md 파일처럼 일반 파일
def read_text_file(path:Path)->str:
    return path.read_text(encoding='utf-8',errors='ignore')

def chunk_text(text:str, chunk_size:int=900, overlab:int=150)->list[str]:
    cleaned = " ".join(text.split())
    if not cleaned:
        return []
    chunks : list[str] = []
    start = 0
    while start < len(cleaned):
        end = min(start+chunk_size, len(cleaned))
        chunks.append(cleaned[start:end])
        if end >= len(cleaned):
            break
        start = end-overlab
    return chunks

from glob import glob
def iter_documents(folder:Path)->Iterable[tuple[Path,str]]:
    for path in folder.rglob('*'):
        if path.is_file():
            surffix = path.suffix.lower()
            if surffix in {'.txt', ".md"}:
                yield path, read_text_file(path)
            elif surffix == '.pdf':
                yield path, read_pdf_file(path)
        
documents_to_add: list[str] = []
metadatas: list[dict[str, str]] = []
ids: list[str] = []

if not enterprise_folder.exists():
    print(f"경고: 기업 문서 폴더가 없습니다: {enterprise_folder}")
    print("폴더 경로를 확인하거나 폴더를 생성한 뒤 다시 실행하세요.")
else:
    for file_path, raw_text in iter_documents(enterprise_folder):
        for index, chunk in enumerate(chunk_text(raw_text), start=1):
            ids.append(f"{file_path.stem}-{index}-{uuid4().hex[:8]}")
            documents_to_add.append(chunk)
            metadatas.append({
                "source_file": file_path.name,
                "source_type": file_path.suffix.lower().lstrip("."),
                "chunk_index": str(index),
            })

    if documents_to_add:
        enterprise_collection.add(
            ids=ids,
            documents=documents_to_add,
            metadatas=metadatas,
        )
        print(f"기업 문서 {len(ids)}개 청크를 벡터DB에 저장했다.")
    else:
        print("enterprise_documents/ 폴더에서 적재할 문서를 찾지 못했다.")

    print("collection count:", enterprise_collection.count()) 

기업 문서 21개 청크를 벡터DB에 저장했다.
collection count: 21


In [23]:
# 생성한 벡터db에서 문장이나 단어를 검색해 보세요
import json
result = enterprise_collection.query(query_texts=['대회 규칙에 대해서 알려주세요'], n_results=2)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
response = client.chat.completions.create(
        model='gpt-5.4-nano',
        messages=[
            {'role':'system','content':'당신은 기업내부또는 외부문서을 기반으로 답변하는 친절한 페르소나 입니다.\n문서에서 추출한 청크들을 기반으로 답변해주세요.'},
            {'role':'user','content': json.dumps(result,ensure_ascii=False)}
        ],
        temperature= 0
    )

print(response.choices[0].message.content.strip())

문서(enterprise.pdf의 18~20 청크) 기준으로 확인되는 내용은 아래와 같습니다.

## 1) 자료 손상/전송·네트워크 문제 등 발생 시 책임 면책 및 참가 제한 가능
- 케이블 연결, 위성 전송, 서버/제공자/컴퓨터 장비의 기술적 고장
- 소프트웨어/하드웨어 고장
- 네트워크 연결 유실 또는 사용 불가능
- 시스템/인적 오류 및 고장
- 인터넷 또는 대회 웹사이트 트래픽 혼잡(또는 이들의 조합)
→ 이런 원인으로 인해 문제가 발생하면 **참가자의 참가가 제한될 수 있음**이 명시되어 있습니다.

## 2) 대회가 계획대로 진행되지 못하는 경우 스폰서의 조치 권한(취소/종료/수정/중지)
- 컴퓨터 바이러스, 버그, 변조, 무단 개입, 사기
- 기술적 오류 또는 행정/보안/공정성/무결성 등에 손상을 주거나 영향을 미치는 기타 원인
- 대회가 계획대로 실행될 수 없는 경우
→ **대회 스폰서는 경품 행사를 취소, 종료, 수정 또는 중지할 권한**이 있습니다.
또한 제출 과정이나 웹사이트를 조작한 참가자는 **실격 처리**할 수 있습니다.

## 3) 웹사이트 의도적 손상/운영 저해 시 법적 책임 및 손해배상 청구 가능
- 대회 웹사이트를 포함해 웹사이트를 의도적으로 손상시키거나
- 대회 스폰서 및 데이콘의 합법적 운영을 저해하려는 시도는
  - 형법 및 민법 위반에 해당할 수 있으며,
→ 그러한 시도가 있으면 **대회 스폰서 및 데이콘이 해당 참가자에게 법률상 최대 범위 내에서 손해배상을 청구**할 수 있다고 되어 있습니다.

## 4) 고용 제안/계약이 아님
- 대회 웹사이트에 별도로 명시되지 않는 한,
- 제출물 제출, 상금 수여, 본 규정의 어떤 내용도
→ **대회 스폰서/주체와의 고용 제안 또는 계약으로 해석되지 않습니다.**
- 참가자는 제출물을 **자발적으로 제출**했으며, 기밀/신탁/대행사/기타 관계 또는 묵시적 계약이 존재하지 않는다고 인정합니다.

## 5) 초상/이름 사용 동의(추가 보상 없이 광고·판촉 목적)
- **스폰서, 데이콘

In [25]:
class EnterpriseRAGState(TypedDict):
    question:str
    retrieved_context:str
    source_items : list[dict[str,str]]
    answer :str
    route:str

def retrieve_with_sources(state:EnterpriseRAGState):
    if enterprise_collection.count == 0:
        return{
            'retrieved_context':'',
            'source_items':[],
            'route':'external'  # 외부문서
        }
    result = enterprise_collection.query(
        query_texts=[state['question']],
        n_results=4,
        include=['documents','metadatas','distances']
    )
    documents = result.get('documents',[[]])[0]
    metadatas = result.get('metadatas',[[]])[0]
    distances = result.get('distances',[[]])[0]
    source_items: list[dict[str, str]] = []
    context_lines: list[str] = []

    for idx, (doc, meta, dist) in enumerate(zip(documents, metadatas, distances), start=1):
        source_id = f"S{idx}"
        file_name = (meta or {}).get("source_file", "unknown")
        chunk_index = (meta or {}).get("chunk_index", "?")
        distance = f"{float(dist):.4f}" if dist is not None else "N/A"

        source_items.append(
            {
                "source_id": source_id,
                "file_name": str(file_name),
                "chunk_index": str(chunk_index),
                "distance": distance,
            }
        )

        context_lines.append(
            f"[{source_id}] file={file_name}, chunk={chunk_index}, distance={distance}\n{doc}"
        )

    if context_lines:
        return {
            "retrieved_context": "\n\n".join(context_lines),
            "source_items": source_items,
            "route": "internal",
        }

    return {
        "retrieved_context": "",
        "source_items": [],
        "route": "external",
    }

def external_reference_dummy(state: EnterpriseRAGState):
    source_items = [
        {
            "source_id": "E1",
            "file_name": "external_policy_reference.md",
            "chunk_index": "1",
            "distance": "N/A",
        },
        {
            "source_id": "E2",
            "file_name": "external_rag_reference.md",
            "chunk_index": "1",
            "distance": "N/A",
        },
    ]
    retrieved_context = (
        "[E1] file=external_policy_reference.md, chunk=1, distance=N/A\n"
        f"질문 '{state['question']}'에 대해 외부 공개 문서의 정책 구조는 일반적으로 목적, 범위, 책임, 절차, 예외 처리로 구성된다.\n\n"
        "[E2] file=external_rag_reference.md, chunk=1, distance=N/A\n"
        "검색된 내부 문서가 없을 때는 외부 공개 자료를 참고해 답변의 뼈대를 만들고, 실제 운영 규정은 반드시 내부 문서로 재검증해야 한다."
    )
    return {
        "retrieved_context": retrieved_context,
        "source_items": source_items,
        "route": "external",
    }


def answer_with_sources(state: EnterpriseRAGState):
    if not state["retrieved_context"].strip():
        return {
            "answer": "검색된 기업 내부 문서가 없어 답변을 생성할 수 없다. 먼저 문서를 벡터DB에 적재해줘.",
        }

    source_labels = ", ".join(item["source_id"] for item in state["source_items"])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "당신은 기업 내부 지식 기반 도우미다. "
                    "반드시 제공된 문맥만 사용해서 답변하고, 핵심 주장 뒤에 출처 라벨을 붙여라. "
                    f"사용 가능한 출처 라벨은 {source_labels} 뿐이다."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"질문: {state['question']}\n\n"
                    f"검색 문맥:\n{state['retrieved_context']}\n\n"
                    "요구사항:\n"
                    "1) 답변 본문에 근거 문장 뒤에 [S1] 같은 출처 라벨을 붙여라.\n"
                    "2) 문맥에 없는 내용은 추측하지 마라."
                ),
            },
        ],
    )
    return {"answer": response.choices[0].message.content.strip()}


enterprise_rag = StateGraph(EnterpriseRAGState)
enterprise_rag.add_node("retrieve_with_sources", retrieve_with_sources)
enterprise_rag.add_node("external_reference_dummy", external_reference_dummy)
enterprise_rag.add_node("answer_with_sources", answer_with_sources)
enterprise_rag.add_edge(START, "retrieve_with_sources")
enterprise_rag.add_conditional_edges(
    "retrieve_with_sources",
    lambda state: state["route"],
    {
        "internal": "answer_with_sources",
        "external": "external_reference_dummy",
    },
)
enterprise_rag.add_edge("external_reference_dummy", "answer_with_sources")
enterprise_rag.add_edge("answer_with_sources", END)
enterprise_rag_app = enterprise_rag.compile()

question = "우리 회사 문서 기준으로 해당 정책의 핵심 절차를 설명해줘"
result = enterprise_rag_app.invoke(
    {
        "question": question,
        "retrieved_context": "",
        "source_items": [],
        "answer": "",
        "route": "",
    }
)

print("question:", question)
print("route:", result.get("route", ""))
print("\nanswer:\n", result["answer"])

print("\n[출처 목록]")
if result["source_items"]:
    for item in result["source_items"]:
        print(
            f"[{item['source_id']}] file={item['file_name']}, chunk={item['chunk_index']}, distance={item['distance']}"
        )
else:
    print("검색된 출처가 없습니다.")

question: 우리 회사 문서 기준으로 해당 정책의 핵심 절차를 설명해줘
route: internal

answer:
 해당 정책의 핵심 절차는 다음과 같습니다:

1. **팀 구성 및 등록**: 팀은 최대 5명으로 구성할 수 있으며, 동일인이 개인 또는 복수팀에 중복 등록할 수 없습니다. 팀 구성 방법은 팀 페이지에서 안내를 확인해야 합니다[S2].

2. **사전학습모델 사용 규칙**: 공식적으로 가중치가 공개된 사전학습모델 중 상업적 또는 비상업적 이용이 허용된 라이선스의 모델만 사용 가능하며, 원격 서버 기반의 API 형태로 접근 가능한 모델은 사용할 수 없습니다[S2].

3. **외부 데이터 사용**: 대회에서 제공하는 학습 데이터 외의 외부 데이터도 사용 가능하지만, 제공된 평가 데이터는 모델 학습에 활용할 수 없습니다[S2][S3].

4. **제출 규칙**: 대회 종료 후 2차 평가 대상자는 코드와 PPT를 정해진 양식에 맞추어 제출해야 하며, 모든 코드는 오류 없이 실행되어야 하고, 상대 경로로 데이터 입/출력 경로를 표기해야 합니다[S2].

5. **데이터 보호 및 비밀 유지**: 대회 참가자는 대회 자료에 접근하지 않은 사람에게 자료를 전송하거나 제공하지 않으며, 승인되지 않은 전송이나 액세스의 가능성을 알게 된 즉시 데이콘에 통지해야 합니다[S3].

6. **법적 준거**: 본 규칙과 관련된 모든 청구는 한국 법의 적용을 받으며, 서울 관할 법원에서 처리됩니다[S4]. 

이러한 절차들은 대회 운영의 공정성과 참가자의 권리를 보호하기 위해 설정되었습니다.

[출처 목록]
[S1] file=enterprise.pdf, chunk=18, distance=0.3536
[S2] file=enterprise.pdf, chunk=4, distance=0.3609
[S3] file=enterprise.pdf, chunk=12, distance=0.3629
[S4] file=enterprise.pdf, chunk=21, distance=0.36

In [27]:
#################################################
# LangGraph StateGraph 상태 설계
#################################################
# openai가 질문의도를 읽고 chromadb가 실제 문서를 검색, 검색 결과를 바탕으로 답변을 생성

####### 상태 설계를 먼저해야 하는 이유 ########
# 질문을 정규화 -> 의도를 추론 -> chromaDB에서 실제 근거를 찾고 -> openai가 그 근거를 바탕으로 답변을 생성


import re
from pathlib import Path
from typing_extensions import TypedDict
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
import os
from sentence_transformers import SentenceTransformer
import chromadb


client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

class StableEmbeddingFunction:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-V2')
    
    def name(self):
        return 'stable_local'        
    
    def __call__(self,input):
        vectors = self.model.encode(
            input,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        return vectors.tolist()

    def embed_query(self,input):
        return self.__call__(input)
    
    
chroma_client = chromadb.PersistentClient()

try:  # 컬렉션은 있으면 생성오류가 발생 그리고 벡터db의 저장할 데이터의 형태가 변경되어도 기존에 존재하면 반영안됨
    chroma_client.delete_collection('test')
except:
    pass

collection = chroma_client.get_or_create_collection(
    name='test',
    embedding_function=StableEmbeddingFunction()
)





# collection 에 데이터 추가
if collection.count() == 0:
    collection.add(
        ids=["doc_langgraph", "doc_react", "doc_rag", "doc_chroma"],
        documents=[
            "LangGraph는 상태 기반 그래프로 다단계 에이전트를 설계한다.",
            "ReAct는 reasoning과 acting을 번갈아 수행해 도구 사용을 결합한다.",
            "RAG는 벡터DB 검색 결과를 근거로 답변 품질을 높인다.",
            "ChromaDB는 문서 임베딩을 저장하고 유사한 문서를 검색하는 벡터DB다.",
        ],
    )   

# 노드에 적용할 함수
# 그래프 노드에 추가
# 엣지적용
# 컴파일  app
# app.invoke --> 결과 
class StateDesignState(TypedDict):
    question: str
    normalized_question: str
    query_intent: str
    retrieved_context: str
    draft_answer: str
    final_answer: str


def normalize_question(state: StateDesignState):
    normalized = re.sub(r"\s+", " ", state["question"].strip().lower())
    return {"normalized_question": normalized}


def infer_intent(state: StateDesignState):
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content": "질문의 의도를 search, explain, compare 중 하나로만 출력한다.",
            },
            {"role": "user", "content": state["normalized_question"]},
        ],
        temperature=0,
    )
    intent = response.choices[0].message.content.strip().lower()
    if intent not in {"search", "explain", "compare"}:
        text = state["normalized_question"]
        if "비교" in text or "차이" in text:
            intent = "compare"
        elif "설명" in text or "왜" in text or "무엇" in text:
            intent = "explain"
        else:
            intent = "search"
    return {"query_intent": intent}


def retrieve_context(state: StateDesignState):
    result = collection.query(query_texts=[state["normalized_question"]], n_results=2)
    context = "\n".join(result["documents"][0])
    return {"retrieved_context": context}


def draft_answer(state: StateDesignState):
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {
                "role": "system",
                "content": "주어진 근거를 바탕으로 3문장 이내의 한국어 답변 초안을 만든다.",
            },
            {
                "role": "user",
                "content": f"질문: {state['question']}\n의도: {state['query_intent']}\n\n근거:\n{state['retrieved_context']}",
            },
        ],
        temperature=0,
    )
    return {"draft_answer": response.choices[0].message.content.strip()}


def finalize_answer(state: StateDesignState):
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content": "초안을 정리하고, 마지막에 한 줄로 이유를 덧붙인다.",
            },
            {
                "role": "user",
                "content": f"질문: {state['question']}\n초안: {state['draft_answer']}",
            },
        ],
        temperature=0,
    )
    return {"final_answer": response.choices[0].message.content.strip()}


workflow = StateGraph(StateDesignState)
workflow.add_node("normalize_question", normalize_question)
workflow.add_node("infer_intent", infer_intent)
workflow.add_node("retrieve_context", retrieve_context)
workflow.add_node("draft_answer", draft_answer)
workflow.add_node("finalize_answer", finalize_answer)

workflow.add_edge(START, "normalize_question")
workflow.add_edge("normalize_question", "infer_intent")
workflow.add_edge("infer_intent", "retrieve_context")
workflow.add_edge("retrieve_context", "draft_answer")
workflow.add_edge("draft_answer", "finalize_answer")
workflow.add_edge("finalize_answer", END)

app = workflow.compile()

result = app.invoke(
    {
        "question": "LangGraph와 RAG의 상태 설계가 왜 중요한가?",
        "normalized_question": "",
        "query_intent": "",
        "retrieved_context": "",
        "draft_answer": "",
        "final_answer": "",
    }
)
print("normalized:", result["normalized_question"])
print("intent:", result["query_intent"])
print("context:")
print(result["retrieved_context"])
print("answer:")
print(result["final_answer"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

normalized: langgraph와 rag의 상태 설계가 왜 중요한가?
intent: explain
context:
LangGraph는 상태 기반 그래프로 다단계 에이전트를 설계한다.
RAG는 벡터DB 검색 결과를 근거로 답변 품질을 높인다.
answer:
LangGraph와 RAG의 상태 설계가 중요한 이유는, 두 방식 모두 “다음 단계가 무엇을 보고/어떻게 판단할지”를 상태가 결정하기 때문입니다.  
- **LangGraph**는 상태 기반 그래프 형태로 다단계 에이전트를 구성하므로, 각 노드(단계)에서 **어떤 정보가 유지되고 어떤 값이 갱신되는지**에 따라 전체 워크플로와 최종 결과가 크게 달라집니다.  
- **RAG**는 벡터DB에서 가져온 **검색 문맥(컨텍스트)**을 근거로 답변을 생성하므로, 그 검색 결과를 상태로 어떻게 저장·정리·우선순위화하느냐에 따라 **정확성, 일관성, 환각 위험**이 달라집니다.  

즉, **상태 설계는 추론 흐름(컨트롤)과 근거(데이터)를 동시에 좌우**하여 시스템의 성능을 결정합니다.  

**이유:** 상태에 담긴 정보의 범위·형식·갱신 규칙이 다음 단계의 판단 근거를 직접 규정하기 때문입니다.
